# 🏛️ ClaimIQ — Week 3A: Databricks Exploration and Bronze Preview

### P19 ClaimIQ · Insurance Risk Analytics — Student Data Engineering Project

```text
FILES → DATAFRAMES → SQL VIEWS → EXPLORATION
→ ONE BRONZE DEMO → LINEAGE
```

The complete Bronze implementation for every source is intentionally reserved for Week 4.
This notebook builds foundational Databricks skills using the ClaimIQ working sources supplied
in the approved Week-2/Week-3 data pack. All data is fully synthetic.


## 🎯 Week-3 outcome

By the end of this notebook, every student should be able to:

- find uploaded files in a Databricks Volume;
- understand how Spark SQL and PySpark read Parquet, JSON and CSV files;
- create temporary views;
- inspect table structure and column types;
- display table contents;
- explain grain and business keys;
- count physical rows versus distinct business keys;
- spot simple, observable data concerns;
- check relationships between claims, policies, policyholders, products, providers and payments;
- create one Bronze demonstration Delta table and one lineage demonstration view.


## 🧩 Databricks cell languages

Databricks allows different cell languages inside one notebook.

Use the cell-language dropdown to choose:

- **Python** for PySpark;
- **SQL** for Spark SQL;
- **File system** for `%fs` commands.

You can also place a magic command at the top of a cell:

```text
%python
%sql
%fs
```

This notebook alternates between short PySpark loading/inspection cells and Spark SQL
analysis cells, matching the style used across the ClaimIQ project.


## 🧭 Notebook map

| Section | What you will do |
|---|---|
| 1 | Check the uploaded files |
| 2 | Create Spark SQL views |
| 3 | Inspect schemas |
| 4 | Display table contents |
| 5 | Understand grain |
| 6 | Count records |
| 7 | Compare physical rows with distinct business keys |
| 8 | Inspect values and ranges |
| 9 | Find simple data concerns |
| 10 | Check relationships between sources |
| 11 | Ask one business question |
| 12 | Preview the Bronze idea (one demo table) |
| 13 | Inspect Delta detail, history and lineage |


# 1. Prepare Databricks

Before running this notebook:

1. Open your Databricks workspace.
2. Attach **Serverless notebook compute**.
3. Run the two setup cells below — the first creates the `claimiq` Volume automatically
   (so you never hit `UC_VOLUME_NOT_FOUND`), the second verifies the six working files.
4. Upload these six Week-3 working files from the approved data pack's `working/` folder into
   the Volume using **Catalog → workspace → default → Volumes → claimiq → Upload**:

```text
claims.parquet
policies.json
policyholders.csv
products.csv
providers.csv
claim_payments.csv
```

Recommended location:

```text
/Volumes/workspace/default/claimiq/
```

> These are the **full working sources**. Never commit the `working/` files to GitHub — only
> the `samples/` extracts belong in `data_sample/raw/`.


## 1.1 Create the Volume automatically

Run this once. `CREATE VOLUME IF NOT EXISTS` is safe to re-run — it does nothing if the
Volume already exists, and creates it if it does not. This removes the most common Week-3
failure: running `%fs ls` against a Volume that was never created
(`[UC_VOLUME_NOT_FOUND] Volume 'workspace'.'default'.'claimiq' does not exist`).


In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.claimiq;

If this cell errors, it means your workspace does not have Unity Catalog enabled on the
`workspace` catalog, or you do not have `CREATE VOLUME` permission on `workspace.default` —
ask your instructor to create the Volume for you, then continue from Section 1.2.


## 1.2 Tip — Know where things live

| Item | Correct place |
|---|---|
| Notebook | Databricks Workspace |
| Full working data files | Unity Catalog Volume |
| Screenshots | GitHub repository |
| Weekly log | GitHub repository |
| Full working dataset | Do not commit to GitHub |

A notebook contains instructions and code. The data itself lives in the Volume, not in GitHub.


# 2. Check the uploaded files

Before loading data, confirm that the six working files are visible in the Volume.

**Upload the six files now** (Section 1, step 4) if you have not already — the check below
will tell you exactly which ones are still missing instead of failing with a raw Spark error
later in the notebook.


In [ ]:
# Defensive check: confirm every required file is present before loading anything.
# This turns a confusing downstream error (e.g. PATH_NOT_FOUND while reading claims.parquet)
# into a clear, actionable message right now.
required_files = [
    "claims.parquet",
    "policies.json",
    "policyholders.csv",
    "products.csv",
    "providers.csv",
    "claim_payments.csv",
]

volume_path = "/Volumes/workspace/default/claimiq"

try:
    present = {f.name.rstrip("/") for f in dbutils.fs.ls(volume_path)}
except Exception as e:
    present = set()
    print(f"Could not list {volume_path}: {e}")
    print("Re-run the 'CREATE VOLUME IF NOT EXISTS' cell above, then retry this cell.")

missing = [f for f in required_files if f not in present]

for f in required_files:
    status = "OK" if f in present else "MISSING"
    print(f"{status:7s} {f}")

if missing:
    print()
    print("Upload the MISSING file(s) above to", volume_path, "before continuing.")
else:
    print()
    print("All six working files are present. Safe to continue.")

In [ ]:
%fs
ls /Volumes/workspace/default/claimiq

### Expected files

```text
claims.parquet
policies.json
policyholders.csv
products.csv
providers.csv
claim_payments.csv
```

If the check above reports anything `MISSING`, stop and upload it before continuing.

> **Professional habit:** Always confirm the source files before writing queries.


# 3. Create Spark SQL views

A Spark SQL view gives a file a simple table-like name.

Instead of repeatedly referring to a long file path, we can write:

```sql
SELECT * FROM claims
```

We will create one temporary view for each of the six source files.


## 3.0 A small loading helper

Each load below now goes through `load_source`, a tiny helper that checks the file exists in
the Volume first. If it does not, you get one clear line telling you which file to upload,
instead of a lower-level Spark error.


In [ ]:
# Small helper used throughout this notebook: loads a file only if it exists,
# and raises a clear, actionable error instead of a raw Spark/Unity Catalog trace.
def load_source(filename, reader):
    """reader is a zero-arg function that performs the actual spark.read.* call."""
    full_path = f"/Volumes/workspace/default/claimiq/{filename}"
    try:
        dbutils.fs.ls(full_path)
    except Exception:
        raise FileNotFoundError(
            f"{filename} was not found at {full_path}. "
            "Upload it to the claimiq Volume (Section 1, step 4) and re-run this cell."
        )
    return reader()

## 3.1 Create the `claims` view

The claims file is Parquet — a compressed, self-describing columnar format, so Spark can read
the schema directly without inference flags.


### What this Python block does

This cell:

1. reads the Parquet claims file;
2. creates a PySpark DataFrame named `claims`;
3. creates a temporary SQL view named `claims`.

Use the cell-language dropdown and select **Python** before running it.


In [ ]:
# Load the claims file as a PySpark DataFrame
claims = load_source(
    "claims.parquet",
    lambda: spark.read.parquet("/Volumes/workspace/default/claimiq/claims.parquet")
)

# Make the DataFrame available to Spark SQL
claims.createOrReplaceTempView("claims")

## 3.2 Create the `policies` view

The policies file is newline-delimited JSON — one policy record per line.


### What this Python block does

This cell reads the policies JSON file and creates both:

- a PySpark DataFrame named `policies`;
- a Spark SQL temporary view named `policies`.


In [ ]:
# Load the policies file
policies = load_source(
    "policies.json",
    lambda: spark.read.json("/Volumes/workspace/default/claimiq/policies.json")
)

# Make it available to Spark SQL
policies.createOrReplaceTempView("policies")

## 3.3 Create the `policyholders` view

### What this Python block does

This cell reads the policyholders CSV and creates both:

- a PySpark DataFrame named `policyholders`;
- a Spark SQL temporary view named `policyholders`.


In [ ]:
# Load the policyholders file
policyholders = load_source(
    "policyholders.csv",
    lambda: spark.read.csv(
        "/Volumes/workspace/default/claimiq/policyholders.csv",
        header=True,
        inferSchema=True
    )
)

# Make it available to Spark SQL
policyholders.createOrReplaceTempView("policyholders")

## 3.4 Create the `products` view

In [ ]:
# Load the products file
products = load_source(
    "products.csv",
    lambda: spark.read.csv(
        "/Volumes/workspace/default/claimiq/products.csv",
        header=True,
        inferSchema=True
    )
)

products.createOrReplaceTempView("products")

## 3.5 Create the `providers` view

In [ ]:
# Load the providers file
providers = load_source(
    "providers.csv",
    lambda: spark.read.csv(
        "/Volumes/workspace/default/claimiq/providers.csv",
        header=True,
        inferSchema=True
    )
)

providers.createOrReplaceTempView("providers")

## 3.6 Create the `claim_payments` view

In [ ]:
# Load the claim payments file
claim_payments = load_source(
    "claim_payments.csv",
    lambda: spark.read.csv(
        "/Volumes/workspace/default/claimiq/claim_payments.csv",
        header=True,
        inferSchema=True
    )
)

claim_payments.createOrReplaceTempView("claim_payments")

## 3.7 Confirm that the views exist

In [ ]:
%sql
SHOW TABLES;

You should see temporary views named:

```text
claims
policies
policyholders
products
providers
claim_payments
```

These views exist for the current notebook session.


# 3A. Confirm the created DataFrames

At this point, six PySpark DataFrames should exist:

```text
claims
policies
policyholders
products
providers
claim_payments
```

Use the following short Python cell to display their names and column counts.


In [ ]:
# Confirm the six DataFrames and their column counts
print("claims columns:", len(claims.columns))
print("policies columns:", len(policies.columns))
print("policyholders columns:", len(policyholders.columns))
print("products columns:", len(products.columns))
print("providers columns:", len(providers.columns))
print("claim_payments columns:", len(claim_payments.columns))

This is a simple existence check.

It does not replace schema inspection, row counts or table previews.


# 4. Inspect the schema

A schema describes the structure of a dataset.

It tells us:

- column names;
- data types;
- possible identifiers;
- date and timestamp fields;
- numeric measures;
- nullable fields.

We will inspect each view separately.


## 4.1 Claims schema

### PySpark method — print the DataFrame schema

This is the quickest way to inspect DataFrame columns and data types.


In [ ]:
# Show claims column names and data types
claims.printSchema()

### Spark SQL method — describe the SQL view

The SQL version displays the same structure in table form.


In [ ]:
%sql
DESCRIBE claims;

### What to notice

Look for:

- `claim_id` — the claims business key;
- `policy_id`, `policyholder_id`, `product_id`, `provider_id` — relationship fields;
- `submission_timestamp`, `review_timestamp`, `decision_timestamp`, `settlement_timestamp`,
  `closure_timestamp` — lifecycle time fields;
- `claim_status`, `outcome_code` — category fields;
- `requested_amount`, `approved_amount`, `reserve_amount`, `deductible_amount` — monetary
  measures;
- `risk_band`, `review_flag` — educational indicators only, not fraud findings.


## 4.2 Policies schema

In [ ]:
%sql
DESCRIBE policies;

## 4.3 Policyholders schema

In [ ]:
%sql
DESCRIBE policyholders;

## 4.4 Products schema

In [ ]:
%sql
DESCRIBE products;

## 4.5 Providers schema

In [ ]:
%sql
DESCRIBE providers;

## 4.6 Claim payments schema

In [ ]:
%sql
DESCRIBE claim_payments;

## 💡 Tip — Compare with the data dictionary

Open `docs/data_dictionary.md` (rewritten from the pack's `data_dictionary.csv`).

Compare it manually with the actual schema.

Ask:

- Are the expected columns present?
- Did Spark detect the expected data types?
- Is any column missing?
- Is any extra column present?
- Does the proposed business key actually exist and look unique?


# 5. Display the table contents

A schema tells us the structure.

The actual rows tell us how the data looks.

Always inspect a few rows before writing analytical queries.


## 5.1 Display claim records

### PySpark method — display DataFrame rows

This uses the `claims` DataFrame created earlier.


In [ ]:
# Display the first 10 claim records
display(claims.limit(10))

### Spark SQL method — display the same rows

In [ ]:
%sql
SELECT *
FROM claims
LIMIT 10;

### Look carefully

Notice:

- how IDs are formatted;
- how timestamps appear, and which ones can be empty;
- the values used in `claim_status` and `outcome_code`;
- whether monetary fields contain decimals or negatives;
- the `risk_band` and `review_flag` values — remember these are educational indicators only.


## 5.2 Display policy records

In [ ]:
# Display the first 10 policy records
display(policies.limit(10))

In [ ]:
%sql
SELECT *
FROM policies
LIMIT 10;

## 5.3 Display policyholder records

In [ ]:
# Display the first 10 policyholder records
display(policyholders.limit(10))

In [ ]:
%sql
SELECT *
FROM policyholders
LIMIT 10;

## 5.4 Display claim payment records

In [ ]:
# Display the first 10 claim payment records
display(claim_payments.orderBy("claim_id", "payment_sequence").limit(10))

In [ ]:
%sql
SELECT *
FROM claim_payments
ORDER BY claim_id, payment_sequence
LIMIT 10;

## 🧠 Student checkpoint 1

Complete these statements:

```text
The main transaction file is ____________________.
One claims row appears to represent ____________________.
The likely claims business key is ____________________.
The main lifecycle status field is ____________________.
The field that links a claim to its payments is ____________________.
```


# 6. Understand the grain

## What is grain?

**Grain means what one row represents.**

For ClaimIQ:

| View | Expected grain |
|---|---|
| `claims` | one physical submitted-claim record; business grain is `claim_id` after DQ |
| `policies` | one physical synthetic policy record; business grain is `policy_id` after DQ |
| `policyholders` | one synthetic policyholder segment record per `policyholder_id` |
| `products` | one synthetic insurance product per `product_id` |
| `providers` | one synthetic service-network provider per `provider_id` |
| `claim_payments` | one physical payment transaction per `payment_id` |

Note the phrase **"business grain after DQ"** for `claims` and `policies`. That is a signal that
the physical row count and the distinct business-key count may not match yet — we test that
next.


# 7. Count the physical records

Start with a simple row count for each view.


## PySpark method — count one DataFrame

This counts the physical rows in the main claims DataFrame.


In [ ]:
# Count physical claim rows
claim_row_count = claims.count()
print("Physical claim rows:", claim_row_count)

## Spark SQL method — count all six views

The SQL query below produces a compact source summary.


In [ ]:
%sql
SELECT 'claims' AS source, COUNT(*) AS records FROM claims
UNION ALL
SELECT 'policies', COUNT(*) FROM policies
UNION ALL
SELECT 'policyholders', COUNT(*) FROM policyholders
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'providers', COUNT(*) FROM providers
UNION ALL
SELECT 'claim_payments', COUNT(*) FROM claim_payments;

This query tells us how many physical rows Spark loaded from each source.

For the claims file, `source_manifest.csv` states the expected physical row count is:

```text
80,240
```

Cross-check every other count you get against `source_manifest.csv` — that file is the
authoritative reconciliation reference for this pack.


# 8. Compare rows with distinct business keys

A physical row count is not always the same as a business record count.

The proposed claims business key is `claim_id`. Let us count distinct claim IDs.


## PySpark method — count distinct business keys

This uses simple DataFrame operations to count unique `claim_id` values.


In [ ]:
# Count distinct claim IDs
distinct_claim_count = (
    claims.select("claim_id")
          .distinct()
          .count()
)

print("Distinct claim IDs:", distinct_claim_count)

## Spark SQL method — compare both values together

The SQL query below is more compact for analytical comparison.


In [ ]:
%sql
SELECT
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT claim_id) AS distinct_claims,
  COUNT(*) FILTER (WHERE claim_id IS NULL) AS null_claim_ids
FROM claims;

### Reading this result

If physical rows are higher than distinct claim IDs, and some `claim_id` values are also
`NULL`, both effects are contributing:

- some claim IDs repeat across physical rows;
- some rows carry no claim ID at all.

> **Professional interpretation:** the physical claims file can contain more rows than there
> are distinct, identifiable claims. Do not assume the difference is caused by only one thing —
> confirm with the queries above and below.


## Do the same check for `policies`

The proposed policies business key is `policy_id`.


In [ ]:
%sql
SELECT
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT policy_id) AS distinct_policies
FROM policies;

## Do the same check for `claim_payments`

The proposed claim-payments business key is `payment_id`.


In [ ]:
%sql
SELECT
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT payment_id) AS distinct_payments,
  COUNT(*) FILTER (WHERE payment_id IS NULL) AS null_payment_ids
FROM claim_payments;

# 9. Display repeated business keys

Now identify a few repeated `claim_id` values.


In [ ]:
%sql
SELECT
  claim_id,
  COUNT(*) AS occurrences
FROM claims
GROUP BY claim_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

## 🧠 Student checkpoint 2

Explain this in one sentence:

```text
The claims physical row count differs from the distinct-claim count because
____________________________________________________________.
```

This is the first grain-related discovery. It belongs in `docs/data_quality_summary.md`
once you have confirmed it with your own query results (do not copy a classmate's numbers —
run the queries yourself).


# 10. Inspect important values

Before looking for errors, understand the normal values in the file.


## 10.1 Claim status distribution

### PySpark method — group and count

This groups the DataFrame by `claim_status` and counts records.


In [ ]:
# Count records by claim status
display(
    claims.groupBy("claim_status")
          .count()
          .orderBy("count", ascending=False)
)

### Spark SQL method — perform the same analysis

In [ ]:
%sql
SELECT
  claim_status,
  COUNT(*) AS records
FROM claims
GROUP BY claim_status
ORDER BY records DESC;

### Why this matters

A distribution helps us understand:

- common categories;
- rare categories;
- possible spelling or casing differences;
- unexpected values;
- whether one status dominates the data.


## 10.2 Loss date range

In [ ]:
%sql
SELECT
  MIN(loss_date) AS earliest_loss_date,
  MAX(loss_date) AS latest_loss_date
FROM claims;

## 10.3 Requested amount range

In [ ]:
%sql
SELECT
  MIN(requested_amount) AS minimum_requested,
  MAX(requested_amount) AS maximum_requested,
  AVG(requested_amount) AS average_requested
FROM claims;

### Look carefully

Does the minimum requested amount look plausible for an insurance claim? A negative minimum is
a strong early signal — keep it in mind for the next section.


## 10.4 Risk band distribution

Remember: `risk_band` is an **educational indicator only**. It is not a fraud finding,
underwriting advice or grounds for any adverse action.


In [ ]:
%sql
SELECT
  risk_band,
  COUNT(*) AS records
FROM claims
GROUP BY risk_band
ORDER BY records DESC;

# 11. Find simple data concerns

Week 3 is about observation.

We are not cleaning the data yet.

We are only asking:

> **What should the Week-4 and Week-5 pipeline handle?**

Keep the eight DQ rule IDs from `dq_requirements.md` (DQ01–DQ08) in mind as you look — each
concern you find below maps to one of them.


## 11.1 Missing claim IDs (maps to DQ01)

In [ ]:
%sql
SELECT COUNT(*) AS missing_claim_ids
FROM claims
WHERE claim_id IS NULL OR TRIM(claim_id) = '';

A missing `claim_id` means the record cannot be trusted at its business grain. Under
DQ01 this is a **Critical** failure — the record must route to `quarantine_claim_records`,
never be silently dropped.


## 11.2 Negative monetary values (maps to DQ05)

In [ ]:
%sql
SELECT COUNT(*) AS negative_monetary_values
FROM claims
WHERE requested_amount < 0
   OR approved_amount < 0
   OR reserve_amount < 0
   OR deductible_amount < 0;

A negative requested, approved, reserve or deductible amount is logically suspicious for
an insurance claim. Under DQ05 this is a **Major** failure.


## 11.3 Approved amount exceeds requested amount without an exception (maps to DQ06)

In [ ]:
%sql
SELECT COUNT(*) AS unexplained_over_approvals
FROM claims
WHERE approved_amount > requested_amount
  AND exception_code IS NULL;

An approved amount above the requested amount is only acceptable when an
`exception_code` documents an approved limit override. Without one, this is a DQ06 concern.


## 11.4 Review timestamp before submission timestamp (maps to DQ04)

In [ ]:
%sql
SELECT COUNT(*) AS review_before_submission
FROM claims
WHERE review_timestamp IS NOT NULL
  AND submission_timestamp IS NOT NULL
  AND review_timestamp < submission_timestamp;

This is an example of an impossible lifecycle sequence: `submitted <= reviewed <=
decision <= paid/closed`.


## 11.5 Display a few suspicious records

In [ ]:
%sql
SELECT
  claim_id,
  submission_timestamp,
  review_timestamp,
  requested_amount,
  approved_amount,
  claim_status
FROM claims
WHERE requested_amount < 0
   OR (review_timestamp IS NOT NULL
       AND submission_timestamp IS NOT NULL
       AND review_timestamp < submission_timestamp)
LIMIT 20;

## 🧠 Student checkpoint 3

Choose one issue and explain:

```text
Issue:
Which DQ rule ID it maps to:
Why it matters:
Which later week should handle it:
```

Suggested answer structure:

> "I found ________. It maps to ________. It could affect ________. It should be handled
> during Silver Candidate / DQ work in Weeks 5–6, and failing records must route to
> `quarantine_claim_records`, not be deleted."


# 12. Check relationships between files

The `claims` file references:

- `policy_id` → `policies`;
- `policyholder_id` → `policyholders`;
- `product_id` → `products`;
- `provider_id` → `providers` (optional — nullable).

A good relationship means the referenced value exists in the related file. This maps to
**DQ02**.


## 12.1 Check the policy relationship

In [ ]:
%sql
SELECT COUNT(*) AS invalid_policy_references
FROM claims c
LEFT JOIN policies p
  ON c.policy_id = p.policy_id
WHERE p.policy_id IS NULL;

These claim rows point to a policy that cannot be found in the `policies` view.


## 12.2 Display a few invalid policy references

In [ ]:
%sql
SELECT
  c.claim_id,
  c.policy_id,
  c.policyholder_id,
  c.claim_status
FROM claims c
LEFT JOIN policies p
  ON c.policy_id = p.policy_id
WHERE p.policy_id IS NULL
LIMIT 20;

## 12.3 Check the policyholder relationship

In [ ]:
%sql
SELECT COUNT(*) AS invalid_policyholder_references
FROM claims c
LEFT JOIN policyholders h
  ON c.policyholder_id = h.policyholder_id
WHERE h.policyholder_id IS NULL;

## 12.4 Check the product relationship

In [ ]:
%sql
SELECT COUNT(*) AS invalid_product_references
FROM claims c
LEFT JOIN products pr
  ON c.product_id = pr.product_id
WHERE pr.product_id IS NULL;

A result of zero is still valuable evidence — it means the relationship was tested, not
skipped.


## 12.5 Check the provider relationship

`provider_id` is nullable — only test it where a value is present.


In [ ]:
%sql
SELECT COUNT(*) AS invalid_provider_references
FROM claims c
LEFT JOIN providers pv
  ON c.provider_id = pv.provider_id
WHERE c.provider_id IS NOT NULL
  AND pv.provider_id IS NULL;

## 12.6 Check the claim_payments → claims relationship

In [ ]:
%sql
SELECT COUNT(*) AS invalid_claim_references_in_payments
FROM claim_payments cp
LEFT JOIN claims c
  ON cp.claim_id = c.claim_id
WHERE c.claim_id IS NULL;

# 13. Understand the join effect

An inner join keeps only matching records.

If some claim rows do not find a policy, those rows will not survive an inner join.


In [ ]:
%sql
SELECT COUNT(*) AS matched_claim_rows
FROM claims c
INNER JOIN policies p
  ON c.policy_id = p.policy_id;

### Reconcile the connection

```text
physical claim rows
−  invalid policy references
= matched claim rows (from the inner join above)
```

Run the arithmetic yourself using the counts you produced in sections 7 and 12.1, and confirm
they agree. This demonstrates why relationship checks matter before building dashboards, and
it previews the Silver-grain reconciliation you will formalize in Weeks 4–6:
`Silver Candidate = Trusted Silver + Quarantine`.


# 14. Ask one business question

A risk analyst asks:

> Which loss category has the highest raw claim volume, and what is its average requested
> amount?

We will group claims by `loss_category` and aggregate.


In [ ]:
%sql
SELECT
  loss_category,
  COUNT(*) AS claim_records,
  COUNT(DISTINCT claim_id) AS distinct_claims,
  ROUND(AVG(requested_amount), 2) AS avg_requested_amount
FROM claims
GROUP BY loss_category
ORDER BY claim_records DESC;

### How to read this result

Write:

```text
Highest-volume loss category:
Number of physical records:
Number of distinct claims:
Average requested amount:
One-line observation:
```

Then add this limitation:

> Known data concerns (Section 11) have not yet been corrected, and quarantine routing has not
> yet been applied, so this is an exploratory result — not a trusted Gold KPI. `risk_band` and
> `review_flag` values shown anywhere in this notebook are educational indicators only, never
> fraud findings or grounds for adverse action.


# 15. Preview the Bronze idea

The Week-3 flow is:

```text
Volume files
→ PySpark DataFrames
→ temporary Spark SQL views
→ exploration
→ one Bronze demonstration table
→ one lineage demonstration view
```

Only one Bronze demonstration table is created here, using the `claims` entity. The complete
multi-table Bronze layer for all six sources belongs to Week 4.


## 15.1 Create one Bronze demo table

This managed Delta table uses the main `claims` entity, preserves the source columns and adds
only basic ingestion metadata. No deduplication, correction or Silver transformation is
performed here.


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.claimiq_week03_bronze_demo_claims
USING DELTA
AS
SELECT
  *,
  current_timestamp() AS ingested_at,
  '/Volumes/workspace/default/claimiq/claims.parquet' AS source_file
FROM claims;

> This is a Week-3 learning table — not the official ClaimIQ Bronze layer.


# 16. Confirm and display the demo table

In [ ]:
%sql
SHOW TABLES IN workspace.default LIKE 'claimiq_week03_bronze_demo_claims';

In [ ]:
%sql
SELECT *
FROM workspace.default.claimiq_week03_bronze_demo_claims
LIMIT 10;

Look for:

```text
ingested_at
source_file
```


# 17. Perform one source-to-demo count check

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM claims) AS source_rows,
  (SELECT COUNT(*) FROM workspace.default.claimiq_week03_bronze_demo_claims) AS demo_rows;

Expected:

```text
source_rows = demo_rows
```

This is a simple Week-3 confidence check. Full reconciliation against DQ and quarantine
belongs to Weeks 5–6.


# 18. Inspect Delta table details

In [ ]:
%sql
DESCRIBE DETAIL workspace.default.claimiq_week03_bronze_demo_claims;

Notice fields such as `format`, `location`, `createdAt`, `lastModified` and `numFiles`.


# 19. Inspect Delta table history

In [ ]:
%sql
DESCRIBE HISTORY workspace.default.claimiq_week03_bronze_demo_claims;

| Concept | Question answered |
|---|---|
| Schema | What columns and data types exist? |
| Relationship | How do business entities connect? |
| History | What operations changed this Delta table? |
| Lineage | Which governed objects feed or use another object? |


# 20. Create a lineage demonstration view

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.default.claimiq_week03_lineage_demo_view
AS
SELECT
  claim_id,
  policy_id,
  policyholder_id,
  product_id,
  provider_id,
  claim_status,
  requested_amount,
  approved_amount,
  ingested_at
FROM workspace.default.claimiq_week03_bronze_demo_claims;

In [ ]:
%sql
SELECT *
FROM workspace.default.claimiq_week03_lineage_demo_view
LIMIT 20;

The governed lineage path is:

```text
claimiq_week03_bronze_demo_claims
                 ↓
claimiq_week03_lineage_demo_view
```


# 21. View lineage in Catalog Explorer

1. Click **Catalog**.
2. Open `workspace`.
3. Open `default`.
4. Select `claimiq_week03_lineage_demo_view`.
5. Open **Lineage**.
6. Choose **See lineage graph** when available.
7. Identify `claimiq_week03_bronze_demo_claims` as the upstream object.
8. Capture a screenshot for your evidence folder.


## 🧠 Student checkpoint

Explain:

```text
Files → DataFrames → temporary views → exploration
→ one Bronze demo table → one lineage demo view
```

Also explain why the complete Bronze layer for all six sources is deferred to Week 4.


# 22. Week-3 boundary

## Completed

- source-file inspection for all six ClaimIQ sources;
- PySpark DataFrame creation and display;
- temporary SQL views;
- schema, grain, counts and values;
- simple data concerns mapped to DQ01–DQ08;
- relationship checks across claims, policies, policyholders, products, providers and payments;
- one business question;
- one Bronze demo table;
- one source-to-demo count check;
- Delta detail and history inspection;
- one lineage demonstration view.

## Deferred to later weeks

- the complete multi-table Bronze layer for all six sources (Week 4);
- Silver Candidate transformation and the full DQ01–DQ08 catalogue (Weeks 5–6);
- Trusted Silver / Quarantine routing and reconciliation (Week 6);
- Gold aggregations for Power BI (Week 7);
- streaming drop processing (Week 10).


# 23. Evidence checklist

```text
screenshots/week03_01_source_files.png
screenshots/week03_02_dataframes.png
screenshots/week03_03_schemas.png
screenshots/week03_04_grain_counts_values.png
screenshots/week03_05_data_concerns.png
screenshots/week03_06_relationship_checks.png
screenshots/week03_07_bronze_demo.png
screenshots/week03_08_delta_detail_history.png
screenshots/week03_09_lineage_graph.png
```

Save these alongside your weekly log in the GitHub repository, not in the Volume.


# 24. Final student defence

Every student should be able to explain:

1. Files, DataFrames and temporary views.
2. Schema, grain and business keys for each of the six sources.
3. Physical rows versus distinct business keys, and why they can differ.
4. Values, ranges and the data concerns found, mapped to DQ rule IDs.
5. Business relationships and joins between claims, policies, policyholders, products,
   providers and payments.
6. What a managed Delta table is, and what `ingested_at` / `source_file` add.
7. Why only one demo Bronze table was built this week, and why the risk_band / review_flag
   fields are educational indicators only — never fraud findings.


# 🎉 Week-3 Databricks foundation complete

```text
Volume files
→ PySpark DataFrames
→ temporary Spark SQL views
→ exploration
→ one Bronze demo table
→ one downstream lineage view
```

This is the correct Week-3 stopping point.

> Week 3 teaches how the ClaimIQ data behaves. Week 4 builds the repeatable Bronze layer for
> all six sources.
